In [1]:
import html
import socket
import subprocess
import threading
import time
import urllib.request

import requests
from flask import Flask, Response, jsonify, request
from IPython.display import display, HTML

!pip install requests

In [2]:
FLASK_PORT = 5000
FLASK_SERVER_URL = f"http://127.0.0.1:{FLASK_PORT}"

HOST = "0.0.0.0"
PORT = 5000
COLAB_GUIDE_URL = "https://research.google.com/colaboratory/faq.html"

# localtunnel error page header
TUNNEL_BYPASS_HEADERS = {
    "bypass-tunnel-window": "true",
    "User-Agent": "curl/8.0.0",
}

STATUS_REFRESH_SEC = 3
PUBLIC_HEALTH_SEC = 30

KEEP_CELL_ALIVE = True

START_TIME = time.time()

In [3]:
# threading share status
STATE = {
    "flask_running":    False,
    "tunnel_alive":     False,
    "public_url":       None,
    "public_health":    "before checking",
    "public_health_at": "-",
    "request_count":    0,
    "last_request":     "-",
    "last_request_at":  "-",
    "password":         None,
    "note":             "initing...",
}

app = Flask(__name__)

In [4]:
import time
import markdown
from flask import Flask, Response

# 필요한 패키지 설치
!pip install requests markdown flask

app = Flask(__name__)

# Colab API 마크다운 가이드 문서
COLAB_GUIDE_TEXT = """
The Colab API lets you programmatically manage Colab runtimes.

The API is currently in beta and available on an allowlist basis. If you're interested in integrating with us, please submit your use cases through this [form](https://docs.google.com/forms/d/e/1FAIpQLSf6q_v7lM7Eqq3PGqMsQBB5pHiuc9XOSsjwJXdT5F2EypmBiA/viewform).

Once access is granted to your project, run `gcloud services enable colaboratory.googleapis.com` to enable the API, and add a `key=<API_KEY>` query parameter to access the API discovery documents below.

We look forward to working with you!

- [REST Resource: v1beta](https://developers.google.com/colab/api/reference/rest#v1beta)
- [REST Resource: v1beta.runtimes](https://developers.google.com/colab/api/reference/rest#v1beta.runtimes)
- [REST Resource: v1beta.runtimespecs](https://developers.google.com/colab/api/reference/rest#v1beta.runtimespecs)
- [REST Resource: v1.operations](https://developers.google.com/colab/api/reference/rest#v1.operations)

## Service: colaboratory.googleapis.com

To call this service, we recommend that you use the Google-provided [client libraries](https://cloud.google.com/apis/docs/client-libraries-explained). If your application needs to use your own libraries to call this service, use the following information when you make the API requests.

### Discovery document

A [Discovery Document](https://developers.google.com/discovery/v1/reference/apis) is a machine-readable specification for describing and consuming REST APIs. It is used to build client libraries, IDE plugins, and other tools that interact with Google APIs. One service may provide multiple discovery documents. This service provides the following discovery documents:

- <https://colaboratory.googleapis.com/$discovery/rest?version=v1>
- <https://colaboratory.googleapis.com/$discovery/rest?version=v1beta>

Discovery document can also be retrieved in [OpenAPI V3.0 format](https://swagger.io/specification/v3/). This service provides the following OpenAPI v3 discovery documents:

- <https://colaboratory.googleapis.com/$discovery/OPENAPI3_0?version=v1>
- <https://colaboratory.googleapis.com/$discovery/OPENAPI3_0?version=v1beta>

### Service endpoint

A [service endpoint](https://cloud.google.com/apis/design/glossary#api_service_endpoint) is a base URL that specifies the network address of an API service. One service might have multiple service endpoints. This service has the following service endpoint and all URIs below are relative to this service endpoint:

- `https://colaboratory.googleapis.com`

## REST Resource: [v1beta](https://developers.google.com/colab/api/reference/rest/v1beta/TopLevel)

| Methods ||
|---|---|
| `https://developers.google.com/colab/api/reference/rest/v1beta/TopLevel/getSubscription` | `GET /v1beta/{name=subscription}` Gets the subscription of the requesting user. |

## REST Resource: [v1beta.runtimes](https://developers.google.com/colab/api/reference/rest/v1beta/runtimes)

| Methods ||
|---|---|
| `https://developers.google.com/colab/api/reference/rest/v1beta/runtimes/create` | `POST /v1beta/runtimes` Creates a Colab runtime assignment. |
| `https://developers.google.com/colab/api/reference/rest/v1beta/runtimes/delete` | `DELETE /v1beta/{name=runtimes/*}` Deletes a Colab runtime assignment. |
| `https://developers.google.com/colab/api/reference/rest/v1beta/runtimes/get` | `GET /v1beta/{name=runtimes/*}` Gets a Colab runtime assigned to the requesting user. |
| `https://developers.google.com/colab/api/reference/rest/v1beta/runtimes/list` | `GET /v1beta/runtimes` Lists Colab runtimes assigned to the requesting user. |

## REST Resource: [v1beta.runtimespecs](https://developers.google.com/colab/api/reference/rest/v1beta/runtimespecs)

| Methods ||
|---|---|
| `https://developers.google.com/colab/api/reference/rest/v1beta/runtimespecs/list` | `GET /v1beta/runtimespecs` Lists available Colab runtime specs and their eligibility to the requesting user. |

## REST Resource: [v1.operations](https://developers.google.com/colab/api/reference/rest/v1/operations)

| Methods ||
|---|---|
| `https://developers.google.com/colab/api/reference/rest/v1/operations/get` | `GET /v1/{name=operations/**}` Gets the latest state of a long-running operation. |
| `https://developers.google.com/colab/api/reference/rest/v1/operations/list` | `GET /v1/operations` Lists operations that match the specified filter in the request. |
| `https://developers.google.com/colab/api/reference/rest/v1/operations/wait` | `GET /v1/{name=operations/*}:wait` Waits until the specified long-running operation is done or reaches at most a specified timeout, returning the latest state. |
"""

def fetch_colab_guide():
    # 마크다운 텍스트를 HTML로 변환 (표 지원을 위해 'tables' 확장 옵션 사용)
    html_body = markdown.markdown(COLAB_GUIDE_TEXT, extensions=['tables'])

    template = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Colab API Guide</title>
    <style>
        body {{
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            line-height: 1.6;
            max-width: 900px;
            margin: 0 auto;
            padding: 20px;
            color: #333;
        }}
        table {{
            border-collapse: collapse;
            width: 100%;
            margin: 20px 0;
        }}
        th, td {{
            border: 1px solid #ddd;
            padding: 8px 12px;
            text-align: left;
        }}
        th {{
            background-color: #f2f2f2;
        }}
        code {{
            background-color: #f4f4f4;
            padding: 2px 4px;
            border-radius: 4px;
            font-family: monospace;
        }}
        pre code {{
            display: block;
            padding: 10px;
        }}
        a {{
            color: #1a73e8;
            text-decoration: none;
        }}
        a:hover {{
            text-decoration: underline;
        }}
    </style>
</head>
<body>
    {html_body}
</body>
</html>"""
    return template

@app.route("/")
def home():
    return Response(fetch_colab_guide(), mimetype="text/html")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


In [5]:
# runtime utils
def is_port_in_use(port: int, host: str = "127.0.0.1") -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0


def serve():
    app.run(host=HOST, port=PORT, threaded=True, use_reloader=False)


# 기존 서버가 있으면 종료 시도
if is_port_in_use(PORT):
    try:
        requests.get(f"{COLAB_GUIDE_URL}/shutdown_flask", timeout=2)
        time.sleep(2)
    except Exception:
        pass

# 백그라운드로 Flask 시작
if not is_port_in_use(PORT):
    threading.Thread(target=serve, daemon=True).start()

# 소켓이 열릴 때까지 대기 (추측 X)
ready_deadline = time.monotonic() + 30
while not is_port_in_use(PORT):
    if time.monotonic() >= ready_deadline:
        raise TimeoutError(f"Flask did not start at {COLAB_GUIDE_URL}")
    time.sleep(0.1)

STATE["flask_running"] = True
STATE["note"] = "Flask 기동 완료. localtunnel 연결 중..."

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


In [6]:
# ---------------------------------------------------------
# 6. 상태판 렌더러
# ---------------------------------------------------------
def render_status_html() -> str:
    elapsed = int(time.time() - START_TIME)
    h, m, s = elapsed // 3600, (elapsed % 3600) // 60, elapsed % 60

    def badge(ok):
        color = "#16a34a" if ok else "#dc2626"
        label = "정상" if ok else "중지"
        return f'<span style="color:{color};font-weight:bold">● {label}</span>'

    flask_ok  = STATE["flask_running"]
    tunnel_ok = STATE["tunnel_alive"]
    pub       = STATE["public_url"] or "(아직 없음)"

    return f"""
    <div style="font-family:ui-monospace,Menlo,Consolas,monospace;font-size:13px;
                line-height:1.7;border:1px solid #d0d7de;border-radius:8px;
                padding:12px 16px;background:#f6f8fa;max-width:860px">
      <div style="font-weight:bold;font-size:14px;margin-bottom:6px">
        🟢 Colab 서버 상태
        <span style="color:#57606a;font-weight:normal">
          (자동 갱신 {STATUS_REFRESH_SEC}초 · 경과 {h:02d}:{m:02d}:{s:02d})
        </span>
      </div>
      <table style="border-collapse:collapse">
        <tr><td style="padding:1px 14px 1px 0">Flask</td>
            <td>{badge(flask_ok)} &nbsp;{COLAB_GUIDE_URL}</td></tr>
        <tr><td style="padding:1px 14px 1px 0">localtunnel</td>
            <td>{badge(tunnel_ok)}</td></tr>
        <tr><td style="padding:1px 14px 1px 0">공개 URL</td>
            <td>{html.escape(pub)}</td></tr>
        <tr><td style="padding:1px 14px 1px 0">공개 /health</td>
            <td>{html.escape(STATE['public_health'])}
                <span style="color:#57606a">(최근 {STATE['public_health_at']})</span></td></tr>
        <tr><td style="padding:1px 14px 1px 0">접속 패스워드</td>
            <td>{html.escape(str(STATE['password']))}</td></tr>
        <tr><td style="padding:1px 14px 1px 0">요청 수</td>
            <td>{STATE['request_count']} 건</td></tr>
        <tr><td style="padding:1px 14px 1px 0">최근 요청</td>
            <td>{html.escape(STATE['last_request'])}
                <span style="color:#57606a">({STATE['last_request_at']})</span></td></tr>
      </table>
      <div style="color:#57606a;margin-top:6px">{html.escape(STATE['note'])}</div>
    </div>
    """


# 상태판을 출력 (display_id 로 이후 계속 업데이트)
status_handle = display(HTML(render_status_html()), display_id="colab_status")


Flask,● 정상 https://research.google.com/colaboratory/faq.html
localtunnel,● 중지
공개 URL,(아직 없음)
공개 /health,before checking (최근 -)
접속 패스워드,None
요청 수,0 건
최근 요청,- (-)


In [7]:
# localtunnel 설치
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [8]:
# 터널 보안 패스워드 확인용 공인 IP
import urllib.request

co_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"🔑 모바일 접속 보안 패스워드: {co_ip}")

🔑 모바일 접속 보안 패스워드: 34.122.90.40


In [9]:
# localtunnel 백그라운드 실행
import subprocess
import time
atus_handle = display(HTML(render_status_html()), display_id="colab_status")

tunnel = subprocess.Popen(
    ["npx", "localtunnel", "--port", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_output = []


def collect_tunnel_output():
    if tunnel.stdout:
        for line in tunnel.stdout:
            tunnel_output.append(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()

public_url = None
url_deadline = time.monotonic() + 120
while time.monotonic() < url_deadline and tunnel.poll() is None:
    for line in tunnel_output:
        for token in line.split():
            candidate = token.strip("'\"(),")
            if candidate.startswith(("https://", "http://")):
                public_url = candidate
                break
        if public_url:
            break
    if public_url:
        break
    time.sleep(0.1)

if public_url is None:
    if tunnel.poll() is None:
        tunnel.terminate()
    raise RuntimeError(
        "localtunnel did not provide a public URL. Output: "
        + "".join(tunnel_output)
    )

STATE["public_url"]  = public_url
STATE["tunnel_alive"] = True
STATE["note"] = "터널 연결 완료. 상태판이 계속 갱신됩니다."

# 접속 패스워드(= 호스트 공인 IP)
try:
    co_ip = urllib.request.urlopen(
        "https://ipv4.icanhazip.com", timeout=10
    ).read().decode("utf8").strip()
    STATE["password"] = co_ip
except Exception as exc:
    STATE["password"] = f"확인 실패: {exc}"

print("🌍 공개 URL :", public_url)
print("🔑 패스워드 :", STATE["password"])

Flask,● 정상 https://research.google.com/colaboratory/faq.html
localtunnel,● 중지
공개 URL,(아직 없음)
공개 /health,before checking (최근 -)
접속 패스워드,None
요청 수,0 건
최근 요청,- (-)


🌍 공개 URL : https://seven-hounds-knock.loca.lt
🔑 패스워드 : 34.122.90.40


In [10]:
# ---------------------------------------------------------
# 8. 백그라운드 모니터 스레드 (핵심)
#    → 셀이 끝나도 상태판을 계속 갱신
# ---------------------------------------------------------
def monitor_loop():
    last_health = 0.0
    while True:
        try:
            # Flask 살아있나?
            STATE["flask_running"] = is_port_in_use(PORT)

            # 터널 살아있나?
            try:
                STATE["tunnel_alive"] = (tunnel.poll() is None)
            except Exception:
                STATE["tunnel_alive"] = False

            # 공개 URL 주기적 헬스체크
            pub = STATE["public_url"]
            if pub and time.time() - last_health >= PUBLIC_HEALTH_SEC:
                last_health = time.time()
                try:
                    r = requests.get(
                        f"{pub.rstrip('/')}/health",
                        timeout=5,
                        headers=TUNNEL_BYPASS_HEADERS,
                    )
                    STATE["public_health"] = (
                        "정상 (200 OK)" if r.ok else f"HTTP {r.status_code}"
                    )
                except requests.RequestException as e:
                    STATE["public_health"] = f"실패: {type(e).__name__}"
                STATE["public_health_at"] = time.strftime("%H:%M:%S")

            # 상태판 갱신
            status_handle.update(HTML(render_status_html()))
        except Exception:
            # 상태판 갱신 실패해도 스레드는 죽이지 않음
            pass

        time.sleep(STATUS_REFRESH_SEC)


threading.Thread(target=monitor_loop, daemon=True).start()

In [ ]:
# ---------------------------------------------------------
# 9. 셀 유지 옵션
#    KEEP_CELL_ALIVE=True  → 셀이 계속 "실행 중" 상태로 남음
#    KEEP_CELL_ALIVE=False → 셀은 끝나지만 상태판은 계속 갱신됨
# ---------------------------------------------------------
if KEEP_CELL_ALIVE:
    print("\n⏹  중단하려면 셀 왼쪽의 ■ (또는 Ctrl/Cmd + M I) 를 누르세요.")
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        STATE["note"] = "사용자가 셀 실행을 중단했습니다."
        print("\n셀 실행이 중단되었습니다. (Flask/터널은 백그라운드에서 계속 동작)")

INFO:werkzeug:127.0.0.1 - - [03/Oct/2026 13:46:11] "GET /health HTTP/1.1" 404 -



⏹  중단하려면 셀 왼쪽의 ■ (또는 Ctrl/Cmd + M I) 를 누르세요.


Flask,● 정상 https://research.google.com/colaboratory/faq.html
localtunnel,● 정상
공개 URL,https://seven-hounds-knock.loca.lt
공개 /health,HTTP 404 (최근 13:47:11)
접속 패스워드,34.122.90.40
요청 수,0 건
최근 요청,- (-)


INFO:werkzeug:127.0.0.1 - - [03/Oct/2026 13:46:23] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Oct/2026 13:46:24] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [03/Oct/2026 13:46:41] "GET /health HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [03/Oct/2026 13:47:11] "GET /health HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [03/Oct/2026 13:47:11] "GET /robots.txt HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [03/Oct/2026 13:47:12] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Oct/2026 13:47:16] "GET /favicon.ico HTTP/1.1" 404 -
